# Proyecto · Clase 6 — Entrenar más rápido y mejor (optimizadores)

Nuestra red ya funciona, pero el descenso por gradiente «simple» baja la montaña a pasos torpes: si `eta` es
chico tarda una eternidad; si es grande, rebota. Hoy probamos dos ideas que usan todos los optimizadores
modernos (Adam incluido) y las **comparamos en una carrera**.

In [ ]:
import os, urllib.request
for archivo in ["digitos.py", "dibujar.py"]:
    if not os.path.exists(archivo):
        urllib.request.urlretrieve("https://raw.githubusercontent.com/datahuba/curso-redes-neuronales/main/proyecto/" + archivo, archivo)

import numpy as np
import matplotlib.pyplot as plt
from digitos import cargar_mnist, ok, guardar_pesos

X_train, y_train, X_test, y_test = cargar_mnist()
X, Xt = X_train.reshape(-1, 784) / 255, X_test.reshape(-1, 784) / 255
Y = np.eye(10)[y_train]

✏️ **Paso 0.** Pega aquí **tus** funciones `relu`, `softmax`, `crear_red`, `adelante` y `gradientes` del
notebook de la Clase 5 (las que ya funcionaban).

In [ ]:
# ✏️ pega aquí tus funciones de la Clase 5

In [ ]:
ok(adelante(crear_red(), X[:3])[2].shape == (3, 10), "tus funciones de la Clase 5 están listas")

## 1. La pista de carreras
`entrenar` recibe un **optimizador**: una función que decide cómo usar las pendientes para mover los pesos.
Para que la carrera sea rápida usamos 20 000 imágenes y medimos el error al final de cada época.

In [ ]:
def error_promedio(red, X, Y):
    return -np.mean(np.sum(Y * np.log(adelante(red, X)[2] + 1e-9), axis=1))


def entrenar(optimizador, epocas=5, lote=64, n=20000, semilla=0):
    red, memoria = crear_red(semilla=semilla), {}
    rng = np.random.default_rng(semilla)
    historia = [error_promedio(red, X[:n], Y[:n])]
    for epoca in range(epocas):
        orden = rng.permutation(n)
        for i in range(0, n, lote):
            idx = orden[i:i + lote]
            optimizador(red, gradientes(red, X[idx], Y[idx]), memoria, epoca)
        historia.append(error_promedio(red, X[:n], Y[:n]))
    return red, historia

## 2. Corredor 1: descenso por gradiente simple (SGD)
Lo que ya hacíamos: `peso ← peso − eta · pendiente`.

In [ ]:
def sgd(red, g, memoria, epoca, eta=0.01):
    for k in red:
        red[k] -= eta * g[k]

## 3. Corredor 2: Momentum (la pelota que rueda)
Una pelota cuesta abajo **acumula velocidad**: si la pendiente apunta siempre hacia el mismo lado, acelera; si
cambia de lado a cada rato (zigzag), los empujones se cancelan.

$$v \leftarrow \beta\, v - \eta \cdot \text{pendiente} \qquad \text{peso} \leftarrow \text{peso} + v$$

`β = 0.9` significa «recuerdo el 90 % de mi velocidad anterior». La velocidad de cada peso se guarda en `memoria`.

✏️ **Ejercicio 1.**

In [ ]:
def momentum(red, g, memoria, epoca, eta=0.01, beta=0.9):
    for k in red:
        v = memoria.get(k, 0)
        v = ...
        memoria[k] = v
        red[k] += ...

## 4. Corredor 3: pasos grandes al principio, chicos al final (decaimiento de la tasa)
Lejos del fondo conviene correr; cerca, caminar con cuidado para no pasarse.

$$\eta_{\text{época}} = \frac{\eta_0}{1 + k \cdot \text{época}}$$

✏️ **Ejercicio 2.**

In [ ]:
def decaimiento(red, g, memoria, epoca, eta0=0.3, k=1.0):
    eta = ...
    for clave in red:
        red[clave] -= eta * g[clave]

In [ ]:
ok(abs(0.3 / (1 + 1.0 * 2) - 0.1) < 1e-12, "fórmula revisada")      # época 2 → eta = 0.1

## 5. ¡Que empiece la carrera! 🏁

In [ ]:
resultados = {}
for nombre, opt in [("SGD (eta 0.01)", sgd), ("Momentum", momentum), ("Decaimiento (eta0 0.3)", decaimiento)]:
    red, historia = entrenar(opt)
    resultados[nombre] = historia
    acc = (adelante(red, Xt)[2].argmax(axis=1) == y_test).mean()
    print(f"{nombre:<24} error final {historia[-1]:.3f}   precisión {acc:.1%}")
    plt.plot(historia, "o-", label=nombre)
plt.xlabel("época"); plt.ylabel("error"); plt.legend(); plt.title("Carrera de optimizadores"); plt.show()

In [ ]:
ok(resultados["Momentum"][-1] < resultados["SGD (eta 0.01)"][-1], "momentum le gana a SGD con el mismo eta")

## 6. Experimentos 🔁
1. SGD con `eta=0.5`, `eta=2.0`: ¿qué le pasa a la curva? (usa `entrenar(lambda r, g, m, e: sgd(r, g, m, e, eta=2.0))`)
2. Momentum con `beta=0.99`. ¿Más memoria es mejor?
3. ¿Cuál elegirías para entrenar la red definitiva? ¿Por qué?

## 7. La red definitiva
Con momentum, todos los datos y más épocas. Después: `python dibujar.py` y compárala con la de la Clase 5.

In [ ]:
def momentum_rapido(red, g, memoria, epoca):
    momentum(red, g, memoria, epoca, eta=0.05, beta=0.9)

red, historia = entrenar(momentum_rapido, epocas=5, n=60000)
precision = (adelante(red, Xt)[2].argmax(axis=1) == y_test).mean()
print(f"Precisión de la red definitiva: {precision:.2%}")
ok(precision > 0.97, "la red definitiva supera el 97 %")
guardar_pesos(**red)

## Lo que nos llevamos
- El **optimizador** decide cómo bajar la montaña; la red y las pendientes son las mismas.
- **Momentum** acelera y suaviza el zigzag. **Decaimiento**: pasos grandes primero, chicos después.
- **Adam**, el que usa casi todo el mundo hoy, combina estas ideas (momentum + un `eta` propio para cada peso).

**Clase 7:** mueve tu dibujo 3 píxeles a un lado… y mira cómo se rompe tu red. La solución: convoluciones.